In [2]:

import re

with open('/workspace/linea_rossa_tracker/src/data/mazzi.ts') as f:
    src = f.read()

DEFAULT = {'nucleare':5,'sanzioni':10,'opinione':0,'defcon':7,'risorse':5,'stabilita':5,
           'risorse_iran':5,'forze_militari_iran':5,'tecnologia_nucleare_iran':5,'stabilita_iran':5,
           'risorse_coalizione':5,'influenza_diplomatica_coalizione':5,'tecnologia_avanzata_coalizione':5,'supporto_pubblico_coalizione':5,'stabilita_coalizione':5,
           'risorse_russia':5,'influenza_militare_russia':5,'veto_onu_russia':2,'stabilita_economica_russia':5,'stabilita_russia':5,
           'risorse_cina':5,'influenza_commerciale_cina':5,'cyber_warfare_cina':5,'stabilita_rotte_cina':5,'stabilita_cina':5,
           'risorse_europa':5,'influenza_diplomatica_europa':5,'aiuti_umanitari_europa':5,'coesione_ue_europa':5,'stabilita_europa':5}

ALL_T = list(DEFAULT.keys())
card_re = re.compile(r"card_id:'([^']+)',\s*card_name:'([^']+)',\s*faction:'([^']+)'.*?effects:\{([^}]+)\}", re.DOTALL)

problems = []
for m in card_re.finditer(src):
    cid, cname, faz, eff_str = m.groups()
    for t in ALL_T:
        fn = re.search(rf'{t}\s*:\s*\(v[^)]*\)\s*=>\s*([^,\}}]+)', eff_str)
        if fn:
            fn_body = fn.group(1).strip().rstrip(',')
            try:
                delta = float(eval(f"(lambda v:{fn_body})(DEFAULT['{t}'])", {'DEFAULT':DEFAULT}))
                if abs(delta) > 5:
                    problems.append({'cid':cid,'cname':cname,'faz':faz,'t':t,'delta':delta,'fn_body':fn_body})
            except Exception as e:
                pass

print(f"Valori anomali (|delta|>5): {len(problems)}")
for p in problems[:50]:
    print(f"  ❌ {p['cid']} '{p['cname']}' [{p['faz']}] {p['t']}: delta={p['delta']} fn='{p['fn_body']}'")


Valori anomali (|delta|>5): 955
  ❌ C025 'Arricchimento Uranio 60%' [Iran] forze_militari_iran: delta=6.0 fn='v+1'
  ❌ C025 'Arricchimento Uranio 60%' [Iran] tecnologia_nucleare_iran: delta=6.0 fn='v+1'
  ❌ C026 'Proxy Hezbollah' [Iran] forze_militari_iran: delta=6.0 fn='v+1'
  ❌ C027 'Diplomazia Regionale' [Iran] stabilita: delta=6.0 fn='v+1'
  ❌ C027 'Diplomazia Regionale' [Iran] risorse_iran: delta=6.0 fn='v+1'
  ❌ C027 'Diplomazia Regionale' [Iran] stabilita_iran: delta=6.0 fn='v+1'
  ❌ C028 'Attacco Droni Houthi' [Iran] forze_militari_iran: delta=6.0 fn='v+1'
  ❌ C029 'Accordo Petrolio Cina' [Iran] risorse: delta=7.0 fn='v+2'
  ❌ C029 'Accordo Petrolio Cina' [Iran] stabilita: delta=6.0 fn='v+1'
  ❌ C029 'Accordo Petrolio Cina' [Iran] influenza_commerciale_cina: delta=6.0 fn='v+1'
  ❌ C030 'Test Missile Balistico' [Iran] forze_militari_iran: delta=6.0 fn='v+1'
  ❌ C031 'Propaganda Press TV' [Iran] stabilita_iran: delta=6.0 fn='v+1'
  ❌ C032 'Centrifughe Avanzate' [Iran] forze_milit

In [5]:

import re

with open('/workspace/linea_rossa_tracker/src/data/mazzi.ts') as f:
    src = f.read()

DEFAULT = {'nucleare':5,'sanzioni':10,'opinione':0,'defcon':7,'risorse':5,'stabilita':5,
           'risorse_iran':5,'forze_militari_iran':5,'tecnologia_nucleare_iran':5,'stabilita_iran':5,
           'risorse_coalizione':5,'influenza_diplomatica_coalizione':5,'tecnologia_avanzata_coalizione':5,'supporto_pubblico_coalizione':5,'stabilita_coalizione':5,
           'risorse_russia':5,'influenza_militare_russia':5,'veto_onu_russia':2,'stabilita_economica_russia':5,'stabilita_russia':5,
           'risorse_cina':5,'influenza_commerciale_cina':5,'cyber_warfare_cina':5,'stabilita_rotte_cina':5,'stabilita_cina':5,
           'risorse_europa':5,'influenza_diplomatica_europa':5,'aiuti_umanitari_europa':5,'coesione_ue_europa':5,'stabilita_europa':5}

ALL_T = list(DEFAULT.keys())

# Analizza i pattern delle funzioni anomale
card_re = re.compile(r"card_id:'([^']+)',\s*card_name:'([^']+)',\s*faction:'([^']+)'.*?effects:\{([^}]+)\}", re.DOTALL)

# Raccoglie i pattern problematici
patterns = {}
for m in card_re.finditer(src):
    cid, cname, faz, eff_str = m.groups()
    for t in ALL_T:
        fn = re.search(rf'{t}\s*:\s*\(v[^)]*\)\s*=>\s*([^,\}}]+)', eff_str)
        if fn:
            fn_body = fn.group(1).strip().rstrip(',')
            try:
                delta = float(eval(f"(lambda v:{fn_body})(DEFAULT['{t}'])", {'DEFAULT':DEFAULT}))
                if abs(delta) > 5:
                    key = fn_body
                    if key not in patterns:
                        patterns[key] = []
                    patterns[key].append((cid, t, delta))
            except: pass

print(f"Pattern problematici unici: {len(patterns)}")
for pat, cards in sorted(patterns.items(), key=lambda x: len(x[1]), reverse=True)[:30]:
    print(f"  fn='{pat}' → usato in {len(cards)} casi, es: {cards[0]}")


Pattern problematici unici: 5
  fn='v+1' → usato in 742 casi, es: ('C025', 'forze_militari_iran', 6.0)
  fn='v+2' → usato in 101 casi, es: ('C029', 'risorse', 7.0)
  fn='v-1' → usato in 82 casi, es: ('C037', 'sanzioni', 9.0)
  fn='v+3' → usato in 19 casi, es: ('NI04', 'risorse', 8.0)
  fn='v-2' → usato in 11 casi, es: ('NR01', 'sanzioni', 8.0)


In [8]:

import re

with open('/workspace/linea_rossa_tracker/src/data/mazzi.ts') as f:
    src = f.read()

DEFAULT = {'nucleare':5,'sanzioni':10,'opinione':0,'defcon':7,'risorse':5,'stabilita':5,
           'risorse_iran':5,'forze_militari_iran':5,'tecnologia_nucleare_iran':5,'stabilita_iran':5,
           'risorse_coalizione':5,'influenza_diplomatica_coalizione':5,'tecnologia_avanzata_coalizione':5,'supporto_pubblico_coalizione':5,'stabilita_coalizione':5,
           'risorse_russia':5,'influenza_militare_russia':5,'veto_onu_russia':2,'stabilita_economica_russia':5,'stabilita_russia':5,
           'risorse_cina':5,'influenza_commerciale_cina':5,'cyber_warfare_cina':5,'stabilita_rotte_cina':5,'stabilita_cina':5,
           'risorse_europa':5,'influenza_diplomatica_europa':5,'aiuti_umanitari_europa':5,'coesione_ue_europa':5,'stabilita_europa':5}

ALL_T = list(DEFAULT.keys())
card_re = re.compile(r"card_id:'([^']+)',\s*card_name:'([^']+)',\s*faction:'([^']+)'.*?effects:\{([^}]+)\}", re.DOTALL)

problems = []
for m in card_re.finditer(src):
    cid, cname, faz, eff_str = m.groups()
    for t in ALL_T:
        fn = re.search(rf'{t}\s*:\s*\(v[^)]*\)\s*=>\s*([^,\}}]+)', eff_str)
        if fn:
            fn_body = fn.group(1).strip().rstrip(',')
            try:
                delta = float(eval(f"(lambda v:{fn_body})(DEFAULT['{t}'])", {'DEFAULT':DEFAULT}))
                if abs(delta) > 5:
                    problems.append({'cid':cid,'cname':cname,'faz':faz,'t':t,'delta':delta,'fn_body':fn_body})
            except: pass

print(f"Valori anomali rimasti (|delta|>5): {len(problems)}")
for p in problems[:30]:
    print(f"  ❌ {p['cid']} '{p['cname']}' [{p['faz']}] {p['t']}: delta={p['delta']} fn='{p['fn_body']}'")


Valori anomali rimasti (|delta|>5): 0


In [11]:

import re, sys

with open('/workspace/linea_rossa_tracker/src/data/mazzi.ts') as f:
    src = f.read()

DEFAULT = {'nucleare':5,'sanzioni':10,'opinione':0,'defcon':7,'risorse':5,'stabilita':5,
           'risorse_iran':5,'forze_militari_iran':5,'tecnologia_nucleare_iran':5,'stabilita_iran':5,
           'risorse_coalizione':5,'influenza_diplomatica_coalizione':5,'tecnologia_avanzata_coalizione':5,'supporto_pubblico_coalizione':5,'stabilita_coalizione':5,
           'risorse_russia':5,'influenza_militare_russia':5,'veto_onu_russia':2,'stabilita_economica_russia':5,'stabilita_russia':5,
           'risorse_cina':5,'influenza_commerciale_cina':5,'cyber_warfare_cina':5,'stabilita_rotte_cina':5,'stabilita_cina':5,
           'risorse_europa':5,'influenza_diplomatica_europa':5,'aiuti_umanitari_europa':5,'coesione_ue_europa':5,'stabilita_europa':5}

state = dict(DEFAULT)
ALL_T = list(DEFAULT.keys())

card_re = re.compile(r"card_id:'([^']+)',\s*card_name:'([^']+)',\s*faction:'([^']+)',\s*card_type:'([^']+)'.*?effects:\{([^}]+)\}", re.DOTALL)
cards = {}
for m in card_re.finditer(src):
    cid, cname, faz, ctype, eff_str = m.groups()
    effects = {}
    for t in ALL_T:
        fn = re.search(rf'{t}\s*:\s*\(v[^)]*\)\s*=>\s*([^,\}}]+)', eff_str)
        if fn:
            fn_body = fn.group(1).strip().rstrip(',')
            try:
                f_fn = eval(f"lambda v:{fn_body}")
                effects[t] = f_fn
            except: pass
    cards[cid] = {'id':cid,'name':cname,'faction':faz,'type':ctype,'effects':effects}

def apply_card(state, card):
    new_state = dict(state)
    deltas = {}
    for t, fn in card['effects'].items():
        if t in state:
            cur = state[t]
            delta = fn(cur)
            if t == 'nucleare': new_val = max(1, min(15, cur + delta))
            elif t == 'sanzioni': new_val = max(1, min(20, cur + delta))
            elif t == 'opinione': new_val = max(-10, min(10, cur + delta))
            elif t == 'defcon': new_val = max(1, min(10, cur + delta))
            else: new_val = max(1, min(12, cur + delta))
            new_state[t] = new_val
            deltas[t] = delta
    return new_state, deltas

test_plays = [
    ('NI34', 'Iran',       'Dichiarazione Soglia Zero — nucleare+, defcon-'),
    ('NC08', 'Coalizione', 'Operazione Congiunta NATO-IDF — nucleare-, tecnologia_nucleare_iran-'),
    ('C033', 'Iran',       'Negoziato JCPOA — sanzioni+, defcon+'),
    ('NE01', 'Europa',     'JCPOA 3.0 UE — nucleare-, influenza_diplomatica_europa+'),
    ('NR40', 'Russia',     'Piano Pace — nucleare-, veto_onu_russia+'),
]

ICON_MAP = {
    'nucleare':'☢️','sanzioni':'🔒','opinione':'📣','defcon':'🚨','risorse':'💵','stabilita':'⚖️',
    'risorse_iran':'💵🇮🇷','forze_militari_iran':'⚔️🇮🇷','tecnologia_nucleare_iran':'🔬🇮🇷','stabilita_iran':'⚖️🇮🇷',
    'risorse_coalizione':'💵🇺🇸','influenza_diplomatica_coalizione':'🤝🇺🇸','tecnologia_avanzata_coalizione':'💻🇺🇸',
    'supporto_pubblico_coalizione':'📢🇺🇸','stabilita_coalizione':'⚖️🇺🇸',
    'risorse_russia':'💵🇷🇺','influenza_militare_russia':'🎖️🇷🇺','veto_onu_russia':'🏛️🇷🇺',
    'stabilita_economica_russia':'📊🇷🇺','stabilita_russia':'⚖️🇷🇺',
    'risorse_cina':'💵🇨🇳','influenza_commerciale_cina':'🏪🇨🇳','cyber_warfare_cina':'🖥️🇨🇳',
    'stabilita_rotte_cina':'🚢🇨🇳','stabilita_cina':'⚖️🇨🇳',
    'risorse_europa':'💵🇪🇺','influenza_diplomatica_europa':'🕊️🇪🇺','aiuti_umanitari_europa':'❤️🇪🇺',
    'coesione_ue_europa':'🌐🇪🇺','stabilita_europa':'⚖️🇪🇺',
}

print("=== SIMULAZIONE 5 TURNI ===\n")
cur_state = dict(state)

for cid, faction, expected in test_plays:
    if cid not in cards:
        print(f"❌ Carta {cid} non trovata")
        continue
    card = cards[cid]
    new_state, deltas = apply_card(cur_state, card)
    print(f"Turno: {faction} gioca '{card['name']}' ({cid})")
    print(f"  Atteso: {expected}")
    non_zero = {k:v for k,v in deltas.items() if v != 0}
    if non_zero:
        for t,d in non_zero.items():
            icon = ICON_MAP.get(t, t[:6])
            before = cur_state.get(t,'?')
            after = new_state.get(t,'?')
            ok = '✅' if d != 0 else '⚠️'
            print(f"  {ok} {icon} {t}: {before} → {after} (Δ{'+' if d>0 else ''}{d})")
    else:
        print(f"  ⚠️ NESSUN EFFETTO APPLICATO")
    print()
    cur_state = new_state

print("=== STATO FINALE TRACCIATI ===")
changes = {k:cur_state[k] for k in ALL_T if cur_state[k] != state[k]}
for k,v in changes.items():
    print(f"  {k}: {state[k]} → {v}")


=== SIMULAZIONE 5 TURNI ===

Turno: Iran gioca 'Dichiarazione Soglia Zero' (NI34)
  Atteso: Dichiarazione Soglia Zero — nucleare+, defcon-
  ✅ ☢️ nucleare: 5 → 7 (Δ+2)
  ✅ 📣 opinione: 0 → 1 (Δ+1)
  ✅ 🚨 defcon: 7 → 5 (Δ-2)
  ✅ ⚔️🇮🇷 forze_militari_iran: 5 → 6 (Δ+1)
  ✅ 🔬🇮🇷 tecnologia_nucleare_iran: 5 → 6 (Δ+1)
  ✅ ⚖️🇺🇸 stabilita_coalizione: 5 → 4 (Δ-1)

Turno: Coalizione gioca 'Operazione Congiunta NATO-IDF' (NC08)
  Atteso: Operazione Congiunta NATO-IDF — nucleare-, tecnologia_nucleare_iran-
  ✅ ☢️ nucleare: 7 → 11 (Δ+4)
  ✅ 🔒 sanzioni: 10 → 12 (Δ+2)
  ✅ 🚨 defcon: 5 → 3 (Δ-2)
  ✅ 💵 risorse: 5 → 3 (Δ-2)
  ✅ ⚔️🇮🇷 forze_militari_iran: 6 → 5 (Δ-1)
  ✅ 🔬🇮🇷 tecnologia_nucleare_iran: 6 → 5 (Δ-1)
  ✅ ⚖️🇮🇷 stabilita_iran: 5 → 4 (Δ-1)
  ✅ 📢🇺🇸 supporto_pubblico_coalizione: 5 → 6 (Δ+1)

Turno: Iran gioca 'Negoziato JCPOA' (C033)
  Atteso: Negoziato JCPOA — sanzioni+, defcon+
  ✅ 🔒 sanzioni: 12 → 14 (Δ+2)
  ✅ 💵 risorse: 3 → 5 (Δ+2)
  ✅ 💵🇮🇷 risorse_iran: 5 → 6 (Δ+1)
  ✅ ⚖️🇮🇷 stabilita_iran: 4 → 5 (Δ+